In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
!pip install -q ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 20.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.0/66.0 kB 3.3 MB/s eta 0:00:00


In [3]:
import torch

print("GPU:", torch.cuda.get_device_name(0))
print("CUDA:", torch.cuda.is_available())

GPU: Tesla T4
CUDA: True


In [4]:
from ultralytics import YOLO

Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.


In [5]:
model = YOLO("yolo11s.pt")

In [7]:
import os

DATASET_DIR = "/kaggle/input/datasets/chaitanyazunzurkar/surveillance-dataset/surveillance_composite"
print(os.listdir(DATASET_DIR))

['labels', 'images', 'surveillance_composite.yaml']


In [8]:
DATA_YAML = f"{DATASET_DIR}/surveillance_composite.yaml"

print(DATA_YAML)

/kaggle/input/datasets/chaitanyazunzurkar/surveillance-dataset/surveillance_composite/surveillance_composite.yaml


In [10]:
import os
import random
import shutil

DATASET = "/kaggle/input/datasets/chaitanyazunzurkar/surveillance-dataset/surveillance_composite"
WORK_DATASET = "/kaggle/working/surveillance_composite"

# Create directories
os.makedirs(f"{WORK_DATASET}/images/train", exist_ok=True)
os.makedirs(f"{WORK_DATASET}/images/val", exist_ok=True)
os.makedirs(f"{WORK_DATASET}/labels/train", exist_ok=True)
os.makedirs(f"{WORK_DATASET}/labels/val", exist_ok=True)

# Get all training images
image_dir = f"{DATASET}/images/train"
label_dir = f"{DATASET}/labels/train"

images = [
    f for f in os.listdir(image_dir)
    if f.lower().endswith((".jpg", ".jpeg", ".png"))
]

print("Total images:", len(images))

# 80/20 split
random.seed(42)
random.shuffle(images)

val_count = int(len(images) * 0.20)

val_images = images[:val_count]
train_images = images[val_count:]

print("Training:", len(train_images))
print("Validation:", len(val_images))

Total images: 6874
Training: 5500
Validation: 1374


In [11]:
# Copy training set
for img in train_images:
    label = os.path.splitext(img)[0] + ".txt"

    shutil.copy(
        f"{image_dir}/{img}",
        f"{WORK_DATASET}/images/train/{img}"
    )

    if os.path.exists(f"{label_dir}/{label}"):
        shutil.copy(
            f"{label_dir}/{label}",
            f"{WORK_DATASET}/labels/train/{label}"
        )

# Copy validation set
for img in val_images:
    label = os.path.splitext(img)[0] + ".txt"

    shutil.copy(
        f"{image_dir}/{img}",
        f"{WORK_DATASET}/images/val/{img}"
    )

    if os.path.exists(f"{label_dir}/{label}"):
        shutil.copy(
            f"{label_dir}/{label}",
            f"{WORK_DATASET}/labels/val/{label}"
        )

print("Train/validation split complete.")

Train/validation split complete.


In [12]:
print("Train images:", len(os.listdir(f"{WORK_DATASET}/images/train")))
print("Val images:", len(os.listdir(f"{WORK_DATASET}/images/val")))
print("Train labels:", len(os.listdir(f"{WORK_DATASET}/labels/train")))
print("Val labels:", len(os.listdir(f"{WORK_DATASET}/labels/val")))

Train images: 5500
Val images: 1374
Train labels: 5382
Val labels: 1347


In [15]:
import os

DATASET = "/kaggle/working/surveillance_composite"
DATA_YAML = f"{DATASET}/surveillance_composite.yaml"

os.makedirs(DATASET, exist_ok=True)

yaml_content = f"""path: {DATASET}

train: images/train
val: images/val

names:
  0: Person
  1: Car
  2: Vehicle
  3: Carrying_object
  4: Bike
  5: Gun
  6: Knife

nc: 7
"""

with open(DATA_YAML, "w") as f:
    f.write(yaml_content)

print("YAML created:")
print(DATA_YAML)
print()
print(yaml_content)

YAML created:
/kaggle/working/surveillance_composite/surveillance_composite.yaml

path: /kaggle/working/surveillance_composite

train: images/train
val: images/val

names:
  0: Person
  1: Car
  2: Vehicle
  3: Carrying_object
  4: Bike
  5: Gun
  6: Knife

nc: 7



In [16]:
import os
import shutil
import random

SOURCE = "/kaggle/input/datasets/chaitanyazunzurkar/surveillance-dataset/surveillance_composite"
DATASET = "/kaggle/working/surveillance_composite"

for d in [
    "images/train",
    "images/val",
    "labels/train",
    "labels/val"
]:
    os.makedirs(f"{DATASET}/{d}", exist_ok=True)

images = [
    f for f in os.listdir(f"{SOURCE}/images/train")
    if f.lower().endswith((".jpg", ".jpeg", ".png"))
]

random.seed(42)
random.shuffle(images)

val_count = int(len(images) * 0.2)

val_images = images[:val_count]
train_images = images[val_count:]

def copy_files(image_list, split):
    count = 0

    for img in image_list:
        label = os.path.splitext(img)[0] + ".txt"

        src_img = f"{SOURCE}/images/train/{img}"
        src_label = f"{SOURCE}/labels/train/{label}"

        if os.path.exists(src_label):
            shutil.copy2(src_img, f"{DATASET}/images/{split}/{img}")
            shutil.copy2(src_label, f"{DATASET}/labels/{split}/{label}")
            count += 1

    return count

print("Copying training data...")
train_count = copy_files(train_images, "train")

print("Copying validation data...")
val_count = copy_files(val_images, "val")

print(f"Train: {train_count}")
print(f"Val:   {val_count}")

Copying training data...
Copying validation data...
Train: 5382
Val:   1347


In [17]:
for split in ["train", "val"]:
    print(
        split,
        "images:", len(os.listdir(f"{DATASET}/images/{split}")),
        "labels:", len(os.listdir(f"{DATASET}/labels/{split}"))
    )

train images: 5500 labels: 5382
val images: 1374 labels: 1347


In [18]:
results = model.train(
    data=DATA_YAML,

    # Training
    epochs=100,
    imgsz=640,
    batch=32,

    # GPU
    device=[0, 1],

    # Checkpoints
    save=True,
    save_period=10,

    # Resume-friendly settings
    project="/kaggle/working/runs",
    name="yolo11s_surveillance",

    # Other
    workers=4,
    pretrained=True,
)

Ultralytics 8.4.131 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
                                                        CUDA:1 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/surveillance_composite/surveillance_composite.yaml, degrees=0.0, deterministic=True, device=0,1, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300